# Data Cleaning -- Training Dataset
**This notebook prepares Flatiron Health data for patients with metastatic RCC in anticipation of training a gradient-boosted survival model to predict mortality from the start of first-line treatment. Specifically, it processes and cleans the training cohort. Prior to data cleaning, the dataset is randomly split into training (80%) and testing (20%) subsets.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorRenal, merge_dataframes

## Split into train and test sets

In [2]:
df = pd.read_csv('../data/LineOfTherapy.csv')

In [3]:
df = (
    df
    .query('LineNumber == 1')
    [['PatientID', 'StartDate']]
)

In [4]:
df.shape

(8337, 2)

In [5]:
processor = DataProcessorRenal()

In [6]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_MetRCCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_MetRCC_Orals.csv',
                                           progression_path = '../data/Enhanced_MetRCCProgression.csv',
                                           drop_dates = False)

2026-03-16 09:59:04,979 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (6345, 2) and unique PatientIDs: 6345
2026-03-16 09:59:04,988 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (8337, 3) and unique PatientIDs: 8337
2026-03-16 09:59:05,292 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-16 09:59:05,299 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (8337, 6) and unique PatientIDs: 8337. There are 0 out of 8337 patients with missing duration values


In [7]:
df = pd.merge(df, mortality_df[['PatientID', 'event']], on = 'PatientID', how = 'left')

In [8]:
df.shape

(8337, 3)

In [9]:
# Stratify split on event 
train, test = train_test_split(
    df,
    test_size = 0.2,
    stratify = df['event'],  
    random_state = 42
)

In [10]:
train.shape

(6669, 3)

In [11]:
test.shape

(1668, 3)

In [12]:
train[['PatientID']].to_csv('../outputs/train_patient_ids.csv', index = False)
test[['PatientID']].to_csv('../outputs/test_patient_ids.csv', index = False)

## Clean CSV files 

In [13]:
train_ids = pd.read_csv('../outputs/train_patient_ids.csv')

In [14]:
train_ids = train_ids.PatientID.to_list()

In [15]:
index_date_df = df[df.PatientID.isin(train_ids)]

In [16]:
index_date_df.shape

(6669, 3)

In [17]:
index_date_df = index_date_df[['PatientID', 'StartDate']]

### Process Enhanced_MetastaticRCC.csv

In [18]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_MetastaticRCC.csv',
                                         index_date_df = index_date_df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2026-03-16 09:59:05,364 - INFO - Successfully read Enhanced_MetastaticRCC.csv file with shape: (10766, 10) and unique PatientIDs: 10766
2026-03-16 09:59:05,369 - INFO - Successfully filtered Enhanced_MetastaticRCC.csv file with shape: (6669, 11) and unique PatientIDs: 6669
2026-03-16 09:59:05,380 - INFO - Successfully processed Enhanced_MetastaticCRC.csv file with final shape: (6669, 13) and unique PatientIDs: 6669


In [19]:
enhanced_df = enhanced_df.copy()

In [20]:
# remove patients with missing MetDiagnosisDate
enhanced_df = enhanced_df.query('MetDiagnosisDate.notna()')

In [21]:
enhanced_df['days_met_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['MetDiagnosisDate']).dt.days
enhanced_df['days_met_to_treatment'] = np.where(enhanced_df['days_met_to_treatment'] < 0 , 0, enhanced_df['days_met_to_treatment'])

enhanced_df['days_diag_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['DiagnosisDate']).dt.days
enhanced_df['days_diag_to_treatment'] = np.where(enhanced_df['days_diag_to_treatment'] < 0 , 0, enhanced_df['days_diag_to_treatment'])

In [22]:
enhanced_df['days_diagnosis_to_met'] = np.where(enhanced_df['days_diagnosis_to_met'] < 0 , 0, enhanced_df['days_diagnosis_to_met'])
enhanced_df['days_diagnosis_to_met'] = enhanced_df['days_diagnosis_to_met'].fillna(0)

In [23]:
enhanced_df.SmokingStatus.value_counts(dropna = False)

SmokingStatus
History of smoking        3811
No history of smoking     2774
Unknown/Not documented      56
Name: count, dtype: int64

In [24]:
enhanced_df['SmokingStatus'] = enhanced_df['SmokingStatus'].map({
    'History of smoking': 1,
    'No history of smoking': 0,
    'Unknown/Not documented': 1
})

In [25]:
enhanced_df.StageFourDetail.value_counts(dropna = False)

StageFourDetail
M1         3414
NaN        3153
T4/M0        69
Unknown       5
Name: count, dtype: int64

In [26]:
enhanced_df['StageFourDetail'] = enhanced_df['StageFourDetail'].fillna('Unknown')

In [27]:
enhanced_df.NephrectomyType_mod.value_counts(dropna = False)

NephrectomyType_mod
Radical nephrectomy    3745
Unknown                2518
Partial nephrectomy     374
Other                     4
Name: count, dtype: int64

In [28]:
enhanced_df['NephrectomyType_mod'] = enhanced_df['NephrectomyType_mod'].map(
    lambda x: 'other/unknown' if x in ['Unknown', 'Other'] else x
)

enhanced_df['NephrectomyType_mod'] = enhanced_df['NephrectomyType_mod'].astype('category')

In [29]:
enhanced_df.Histology.value_counts()

Histology
Clear Cell                  4652
RCC, NOS                    1267
Papillary                    505
Chromophobe                  105
Other                         59
Translocation                 16
Renal Medullary               12
Non-Clear Cell Carcinoma       2
Name: count, dtype: int64

In [30]:
histology_mapping = {
    'Clear Cell': 'clear_cell',
    'RCC, NOS': 'nos',  
    'Papillary': 'papillary',
    'Chromophobe': 'chromophobe',
    'Collecting Duct': 'rare_aggressive',
    'Renal Medullary': 'rare_aggressive',  
    'Translocation': 'rare_other',
    'Other': 'rare_other',
    'Non-Clear Cell Carcinoma': 'rare_other'
}

enhanced_df['Histology_grouped'] = enhanced_df['Histology'].map(histology_mapping).astype('category')
enhanced_df = enhanced_df.drop(columns = ['Histology'])

In [31]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'MetDiagnosisDate',
                                          'NephrectomyDate',
                                          'imported_StartDate'])

### Process Demographics.csv 

In [32]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2026-03-16 09:59:05,438 - INFO - Successfully read Demographics.csv file with shape: (10766, 6) and unique PatientIDs: 10766
2026-03-16 09:59:05,445 - WARNING - Found 1 ages outside valid range (18-120)
2026-03-16 09:59:05,452 - INFO - Successfully processed Demographics.csv file with final shape: (6669, 6) and unique PatientIDs: 6669


In [33]:
demographics_df = demographics_df.copy()

In [34]:
demographics_df = demographics_df.query('age >= 18 and age <= 120', engine = 'python')

In [35]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M    4701
F    1967
Name: count, dtype: int64

In [36]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [37]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_MetRCCBiomarkers.csv

In [38]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_MetRCCBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-03-16 09:59:05,473 - INFO - Successfully read Enhanced_MetRCCBiomarkers.csv file with shape: (878, 17) and unique PatientIDs: 692
2026-03-16 09:59:05,479 - INFO - Successfully merged Enhanced_MetRCCBiomarkers.csv df with index_date_df resulting in shape: (669, 18) and unique PatientIDs: 524
2026-03-16 09:59:05,496 - INFO - Successfully processed Enhanced_MetRCCBiomarkers.csv file with final shape: (6669, 3) and unique PatientIDs: 6669


In [39]:
biomarkers_df.PDL1_percent_staining.value_counts(dropna = False)

PDL1_percent_staining
NaN          6635
5% - 9%         9
10% - 19%       4
50% - 59%       4
80% - 89%       4
20% - 29%       3
100%            3
90% - 99%       2
1%              1
30% - 39%       1
40% - 49%       1
60% - 69%       1
70% - 79%       1
0%              0
< 1%            0
2% - 4%         0
Name: count, dtype: int64

In [40]:
def map_pdl1(value):
    if pd.isna(value):  # leave missing as is
        return value
    elif value in ['0%', '< 1%']:
        return '0%'
    else:
        return '>=1%'

biomarkers_df['PDL1_binary'] = biomarkers_df['PDL1_percent_staining'].apply(map_pdl1)

In [41]:
biomarkers_df.PDL1_binary.value_counts(dropna = False)

PDL1_binary
NaN     6635
>=1%      34
Name: count, dtype: int64

In [42]:
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

In [43]:
biomarkers_df = biomarkers_df.drop(columns = ['PDL1_percent_staining'])

### Process ECOG.csv

In [44]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-03-16 09:59:05,562 - INFO - Successfully read ECOG.csv file with shape: (141710, 4) and unique PatientIDs: 7722
2026-03-16 09:59:05,598 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (105860, 5) and unique PatientIDs: 5025
2026-03-16 09:59:05,672 - INFO - Successfully processed ECOG.csv file with final shape: (6669, 3) and unique PatientIDs: 6669


In [45]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
NaN    2934
0      1554
1      1469
2       543
3       155
4        14
Name: count, dtype: int64

In [46]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 0 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(0)

In [47]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [48]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-03-16 09:59:07,583 - INFO - Successfully read Vitals.csv file with shape: (2216004, 16) and unique PatientIDs: 10743
2026-03-16 09:59:08,757 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (1592953, 17) and unique PatientIDs: 6663
2026-03-16 09:59:09,571 - INFO - Successfully processed Vitals.csv file with final shape: (6669, 8) and unique PatientIDs: 6669


### Process Lab.csv

In [49]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'neutrophil': ['26499-4', '751-8', '753-4']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-03-16 09:59:18,281 - INFO - Successfully read Lab.csv file with shape: (7341289, 17) and unique PatientIDs: 10244
2026-03-16 09:59:21,154 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (5391702, 18) and unique PatientIDs: 6492
2026-03-16 09:59:32,553 - INFO - Successfully processed Lab.csv file with final shape: (6669, 81) and unique PatientIDs: 6669


### Process MedicationAdministration.csv

In [50]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-03-16 09:59:33,126 - INFO - Successfully read MedicationAdministration.csv file with shape: (407234, 11) and unique PatientIDs: 7413
2026-03-16 09:59:33,295 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (283933, 12) and unique PatientIDs: 5363
2026-03-16 09:59:33,334 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (6669, 9) and unique PatientIDs: 6669


### Process Diagnosis.csv

In [51]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-03-16 09:59:33,558 - INFO - Successfully read Diagnosis.csv file with shape: (342031, 6) and unique PatientIDs: 10766
2026-03-16 09:59:33,632 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (235968, 7) and unique PatientIDs: 6669
2026-03-16 09:59:34,275 - INFO - Successfully processed Diagnosis.csv file with final shape: (6669, 40) and unique PatientIDs: 6669


In [52]:
diagnosis_df['gi_met_combined'] = (
    diagnosis_df['adrenal_met'] | diagnosis_df['peritoneum_met'] | diagnosis_df['gi_met']
)

diagnosis_df = diagnosis_df.drop(columns = ['adrenal_met', 'peritoneum_met', 'gi_met'])

### Process Enhanced_Mortality_V2.csv

In [53]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_MetRCCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_MetRCC_Orals.csv',
                                           progression_path = '../data/Enhanced_MetRCCProgression.csv',
                                           drop_dates = False)

2026-03-16 09:59:34,289 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (6345, 2) and unique PatientIDs: 6345
2026-03-16 09:59:34,298 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (6669, 3) and unique PatientIDs: 6669
2026-03-16 09:59:34,582 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-16 09:59:34,590 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (6669, 6) and unique PatientIDs: 6669. There are 0 out of 6669 patients with missing duration values


In [54]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [55]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [56]:
training_df = merge_dataframes(enhanced_df,
                               demographics_df,
                               biomarkers_df,
                               ecog_df,
                               vitals_df,
                               labs_df,
                               medications_df,
                               diagnosis_df,
                               mortality_df,
                               merge_type = 'inner')

2026-03-16 09:59:34,607 - INFO - Anticipated number of merges: 8
2026-03-16 09:59:34,607 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 155
2026-03-16 09:59:34,608 - INFO - Dataset 1 shape: (6641, 11), unique PatientIDs: 6641
2026-03-16 09:59:34,609 - INFO - Dataset 2 shape: (6668, 6), unique PatientIDs: 6668
2026-03-16 09:59:34,610 - INFO - Dataset 3 shape: (6669, 3), unique PatientIDs: 6669
2026-03-16 09:59:34,611 - INFO - Dataset 4 shape: (6669, 4), unique PatientIDs: 6669
2026-03-16 09:59:34,611 - INFO - Dataset 5 shape: (6669, 8), unique PatientIDs: 6669
2026-03-16 09:59:34,612 - INFO - Dataset 6 shape: (6669, 81), unique PatientIDs: 6669
2026-03-16 09:59:34,613 - INFO - Dataset 7 shape: (6669, 9), unique PatientIDs: 6669
2026-03-16 09:59:34,613 - INFO - Dataset 8 shape: (6669, 38), unique PatientIDs: 6669
2026-03-16 09:59:34,614 - INFO - Dataset 9 shape: (6657, 3), unique PatientIDs: 6657
2026-03-16 09:59:34,617 - 

In [57]:
training_df.shape

(6628, 155)

## Export dataframe

In [58]:
training_df.to_csv('../outputs/1L_features_training.csv', index = False)

In [59]:
# Save dtypes
training_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_training_dtypes.csv')